# Prithvi flood-extent inference

This notebook is limited to the model workflow: Sentinel-2 preparation, Prithvi-100M-sen1floods11 inference, validation, GeoJSON conversion, and a provenance-complete `HazardEvent` fixture.

**Scope:** water/flood segmentation only. Do not interpret this model output as landslide detection or operational flood verification.

**Input band order:** Blue (B02), Green (B03), Red (B04), Narrow NIR (B8A), SWIR1 (B11), SWIR2 (B12). The model expects a single six-band GeoTIFF in exactly this order.

In [11]:
!python3.9 -c "import sys; print(sys.version)"
!python3.9 -c "import ipykernel; print('IPYKERNEL OK')"

3.9.25 (main, Nov  7 2025, 18:07:57) 
[GCC 11.4.0]
IPYKERNEL OK


## 0. Runtime requirements

The original Prithvi-100M Sen1Floods11 inference stack uses the legacy MMSegmentation/MMCV APIs. Run this notebook in a fresh **Python 3.9** GPU environment (for example, a conda environment); do not mix it with TerraTorch or the newer Prithvi-EO-2.0 stack. The preflight below intentionally stops on a different Python version, because substituting incompatible versions can silently invalidate the run.

In [18]:
import sys
from pathlib import Path

assert sys.version_info[:2] == (3, 9), (
    'Use a fresh Python 3.9 GPU environment for this legacy Prithvi-100M inference stack.'
)
WORKDIR = Path('/content/prithvi_flood')
WORKDIR.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version)
print('Working directory:', WORKDIR)

AssertionError: Use a fresh Python 3.9 GPU environment for this legacy Prithvi-100M inference stack.

## 1. Install the official legacy inference dependencies

This follows the NASA IMPACT `hls-foundation-os` inference path used by the model card. Run once in the fresh environment, then restart the kernel if the installer asks.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/NASA-IMPACT/hls-foundation-os.git
%cd /content/hls-foundation-os
!pip install -q -e .
!pip install -q -U openmim
# Select the wheel URL that matches the CUDA and torch versions in this environment.
!mim install 'mmcv-full==1.6.2' -f https://download.openmmlab.com/mmcv/dist/cu115/torch1.11.0/index.html
!pip install -q 'mmsegmentation==0.30.0' rasterio shapely matplotlib huggingface_hub

## 2. Locate and unpack the Sentinel-2 L2A scene

Set `SCENE_ZIP_PATH` to the uploaded `.SAFE.zip` file. The notebook discovers the six required band files rather than relying on pasted paths.

In [ ]:
# Example for Colab Drive. Replace this path when running locally.
SCENE_ZIP_PATH = Path('/content/drive/MyDrive/terracascade/S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431.SAFE.zip')
assert SCENE_ZIP_PATH.exists(), f'Missing scene archive: {SCENE_ZIP_PATH}'

import shutil
SCENE_DIR = WORKDIR / 'scene'
if not SCENE_DIR.exists():
    shutil.unpack_archive(SCENE_ZIP_PATH, WORKDIR)
    extracted = next(WORKDIR.glob('*.SAFE'))
    extracted.rename(SCENE_DIR)

def find_one(pattern):
    matches = list(SCENE_DIR.rglob(pattern))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {pattern}; found {len(matches)}: {matches}')
    return matches[0]

band_paths = {
    'B02': find_one('*_B02_10m.jp2'), 'B03': find_one('*_B03_10m.jp2'),
    'B04': find_one('*_B04_10m.jp2'), 'B8A': find_one('*_B8A_20m.jp2'),
    'B11': find_one('*_B11_20m.jp2'), 'B12': find_one('*_B12_20m.jp2'),
}
band_paths

## 3. Build a six-band GeoTIFF for the AOI

Enter the AOI bounds in WGS84. Crop before inference: a full Sentinel-2 tile is unnecessarily large and can exhaust GPU memory. The output is explicitly written with the GeoTIFF driver; copying a JP2 profile was the cause of the previous write failure.

In [ ]:
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds, reproject

# Replace with the reviewed AOI boundary: west, south, east, north (EPSG:4326).
AOI_NAME = 'Idamalayar AOI'
AOI_BOUNDS_WGS84 = (76.65, 10.14, 76.80, 10.36)
BAND_ORDER = ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12']

with rasterio.open(band_paths['B02']) as ref_src:
    bounds = transform_bounds('EPSG:4326', ref_src.crs, *AOI_BOUNDS_WGS84, densify_pts=21)
    ref_window = from_bounds(*bounds, transform=ref_src.transform).round_offsets().round_lengths()
    ref_transform = ref_src.window_transform(ref_window)
    ref_crs = ref_src.crs
    height, width = int(ref_window.height), int(ref_window.width)
    assert height > 0 and width > 0, 'AOI does not overlap the Sentinel-2 scene'

stack = np.zeros((6, height, width), dtype=np.float32)
for i, band in enumerate(BAND_ORDER):
    with rasterio.open(band_paths[band]) as src:
        reproject(
            source=rasterio.band(src, 1), destination=stack[i],
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref_transform, dst_crs=ref_crs,
            dst_shape=(height, width), resampling=Resampling.bilinear,
        )

STACKED_TIF = WORKDIR / 'idamalayar_6band_input.tif'
profile = {'driver': 'GTiff', 'height': height, 'width': width, 'count': 6, 'dtype': 'float32',
           'crs': ref_crs, 'transform': ref_transform, 'compress': 'deflate'}
with rasterio.open(STACKED_TIF, 'w', **profile) as dst:
    dst.write(stack)
    dst.descriptions = tuple(BAND_ORDER)
print(STACKED_TIF, stack.shape, BAND_ORDER)

In [ ]:
import matplotlib.pyplot as plt
rgb = stack[[2, 1, 0]]
rgb = np.clip(rgb / np.percentile(rgb, 98), 0, 1).transpose(1, 2, 0)
plt.figure(figsize=(8, 8)); plt.imshow(rgb); plt.title('RGB band-order check'); plt.axis('off');
assert rasterio.open(STACKED_TIF).count == 6

## 4. Download the exact model files and run the official inference script

The legacy repository name redirects to `ibm-nasa-geospatial/Prithvi-EO-1.0-100M-sen1floods11`. Save the resolved repository and file paths in the provenance record. The model output labels are 0 = no water, 1 = water/flood, -1 = no data/cloud.

In [ ]:
from huggingface_hub import hf_hub_download
MODEL_REPOSITORY = 'ibm-nasa-geospatial/Prithvi-EO-1.0-100M-sen1floods11'
config_path = hf_hub_download(MODEL_REPOSITORY, 'sen1floods11_Prithvi_100M.py')
checkpoint_path = hf_hub_download(MODEL_REPOSITORY, 'sen1floods11_Prithvi_100M.pth')
print(config_path, checkpoint_path, sep='\n')

INPUT_DIR, OUTPUT_DIR = WORKDIR / 'inference_input', WORKDIR / 'inference_output'
INPUT_DIR.mkdir(exist_ok=True); OUTPUT_DIR.mkdir(exist_ok=True)
shutil.copy2(STACKED_TIF, INPUT_DIR / STACKED_TIF.name)
%cd /content/hls-foundation-os
!python model_inference.py -config {config_path} -ckpt {checkpoint_path} -input {INPUT_DIR} -output {OUTPUT_DIR} -input_type tif -bands 0 1 2 3 4 5
print(list(OUTPUT_DIR.rglob('*')))

## 5. Validate the mask, polygonise flood pixels, and create the fixture

Set `MASK_TIF` to the raster emitted by `model_inference.py`. Inspect the overlay before accepting the result. Remove very small polygons to avoid pixel-scale noise; this threshold is a demo simplification, not model confidence.

In [ ]:
from datetime import datetime, timezone
import json
from shapely.geometry import shape, mapping
from rasterio.features import shapes

MASK_TIF = next(OUTPUT_DIR.rglob('*.tif'))  # verify this is the predicted-label raster, not a visualisation
with rasterio.open(MASK_TIF) as src:
    mask = src.read(1)
    assert mask.shape == (height, width), (mask.shape, (height, width))

plt.figure(figsize=(8, 8)); plt.imshow(rgb); plt.imshow(np.ma.masked_where(mask != 1, mask), cmap='Blues', alpha=.55); plt.title('Prithvi water/flood mask overlay'); plt.axis('off');

MIN_POLYGON_AREA_M2 = 2_500
features = []
for geometry, value in shapes((mask == 1).astype('uint8'), mask=(mask == 1), transform=ref_transform):
    polygon = shape(geometry)
    if polygon.area >= MIN_POLYGON_AREA_M2:
        feature_id = f'flood-zone-{len(features) + 1:03d}'
        features.append({'type': 'Feature', 'id': feature_id, 'properties': {'id': feature_id}, 'geometry': mapping(polygon)})

flood_extent = {'type': 'FeatureCollection', 'features': features}
assert features, 'No polygons remain: verify the predicted mask and AOI before continuing.'
(WORKDIR / 'flood_extent.geojson').write_text(json.dumps(flood_extent, indent=2))
print(f'{len(features)} flood polygons written')

In [ ]:
SCENE_ID = 'S2B_MSIL2A_20260810T050649_N0512_R019_T43PGM_20260810T085431'
SCENE_ACQUIRED_AT = '2026-08-10T05:06:49Z'
RUN_AT = datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')
hazard_event = {
    'id': 'flood-idamalayar-20260810', 'hazard': 'flood', 'severity': 'orange',
    'source': f'Prithvi-100M-sen1floods11 inference ({MODEL_REPOSITORY}), Sentinel-2 L2A scene {SCENE_ID} acquired {SCENE_ACQUIRED_AT}, {AOI_NAME}',
    'status': 'verified-demo', 'issuedAt': SCENE_ACQUIRED_AT,
    'affectedZones': [feature['id'] for feature in features],
    'limitations': ['single-timestamp inference', 'not a live feed', 'demo AOI only', 'cloud-sensitive optical input; not ground-truthed'],
}
provenance = {
    'model': 'Prithvi-100M-sen1floods11', 'resolvedModelRepository': MODEL_REPOSITORY,
    'configPath': str(config_path), 'checkpointPath': str(checkpoint_path),
    'inputBands': BAND_ORDER, 'sceneId': SCENE_ID, 'sceneAcquiredAt': SCENE_ACQUIRED_AT,
    'aoiName': AOI_NAME, 'aoiBoundsWgs84': AOI_BOUNDS_WGS84, 'runAt': RUN_AT,
    'maskFile': str(MASK_TIF), 'minimumPolygonAreaM2': MIN_POLYGON_AREA_M2,
}
fixture = {'hazardEvent': hazard_event, 'floodExtent': flood_extent, 'provenance': provenance}
FIXTURE_PATH = WORKDIR / 'hazard_event_fixture.json'
FIXTURE_PATH.write_text(json.dumps(fixture, indent=2))
print(FIXTURE_PATH)

## Outputs

- `flood_extent.geojson`: model-derived water/flood polygons.
- `hazard_event_fixture.json`: the `HazardEvent`, GeoJSON, and full provenance record.

Only accept and distribute these files after the mask overlay has been manually checked for incorrect band order, cloud/no-data artefacts, and AOI alignment.